# FarmTech Solutions · Fase 6 · **Ir Além 3.1**
## Coleta de imagens por câmera Wi-Fi e reconhecimento em tempo real com o `best.pt`

**Aluno:** Jefferson Gonçalves Lemos · RM 572399

| Material | Link |
|---|---|
| Vídeo de demonstração (até 5 min, não listado) | ⏳ `https://youtu.be/<inserir>` |
| Vídeo complementar sobre a escolha do Hailo | ⏳ `https://youtu.be/<inserir>` |
| Notebook principal (Entregas 1 e 2) | `../JeffersonGoncalvesLemos_rm572399_pbl_fase6.ipynb` |

Neste notebook documento a opção 3.1 do Ir Além. Levei o módulo de visão do **rover autônomo de pasto**
da FarmTech (classes `vaca`, `cobra`, `escorpiao`, `arvore` e `cerca`) para fora do Colab: uma câmera
coleta as imagens e as transmite por Wi-Fi, e um programa Python executado no VS Code reconhece os
objetos com o modelo que treinei na Entrega 1.

## 1. O que o enunciado pede e como atendi

A opção 3.1 do enunciado aceita uma **ESP32-CAM real** ou a **webcam do PC conectada ao Python lendo o `best.pt`**,
e descreve o fluxo: *coletar imagens em tempo real e transmitir para um Visual Code que reconheça os objetos*.

Organizei a solução em **dois modos que usam o mesmo cliente**:

- **BASE (o fluxo descrito no enunciado):** a câmera (ESP32-CAM, Raspberry Pi 5 ou webcam) coleta as
  imagens e as transmite como MJPEG por Wi-Fi; o `cliente_vscode.py`, executado no VS Code do PC,
  carrega o `best.pt` e detecta os objetos. A webcam do PC também funciona como fonte local (`--fonte 0`).
- **PLUS (extensão do que foi pedido):** a inferência sai do PC e passa para dentro do rover. O
  Raspberry Pi 5 com a NPU **Hailo-8L** executa o mesmo modelo, compilado em `.hef`, e envia por Wi-Fi o
  vídeo já anotado e as detecções em JSON. Como plano B, o mesmo modelo roda em ONNX Runtime na CPU do Pi.

A tabela relaciona cada requisito do enunciado com a parte do projeto que o atende.

| Requisito do enunciado | Onde está |
|---|---|
| Coleta e envio das imagens | `camera_server.py` (Pi/PC) e `esp32cam/esp32cam_stream.ino` (ESP32-CAM) → `cliente_vscode.py` |
| Integração Wi-Fi estável e eficiente | MJPEG sobre HTTP, reconexão automática, RTT e latência medidos (seções 6 e 7) |
| Escolha e justificativa do hardware | Seção 2 |
| Código organizado e comentado | Pasta `ir_alem/`, seção 4 |
| Figura da arquitetura | Seção 3 (`figuras/arquitetura.png`, de autoria própria) |
| Evidências e vídeo | Seções 7 e 8 |

## 2. Justificativa das escolhas

### Câmera + Wi-Fi em vez de processar tudo no PC com um cabo
O rover se desloca pelo pasto, então a câmera precisa estar **nele**, longe do computador. Escolhi o
Wi-Fi porque é o enlace que já existe na sede da fazenda e não exige infraestrutura nova. O custo é uma
latência variável e eventuais quedas de sinal; por isso o cliente reconecta sozinho e registra a latência
de cada quadro.

### MJPEG sobre HTTP
- É o formato nativo da ESP32-CAM: o sensor OV2640 já entrega JPEG e o microcontrolador apenas repassa os bytes.
- Abre em qualquer navegador, no VLC e no OpenCV, o que simplifica a depuração: basta abrir uma URL.
- Cada quadro é independente. Um pacote perdido compromete apenas um quadro, e não um trecho do vídeo,
  como acontece com H.264. Em troca, cada quadro consome mais banda.

### ESP32-CAM × Raspberry Pi 5 × Pi 5 + Hailo-8L

| | ESP32-CAM | Raspberry Pi 5 | Pi 5 + Hailo-8L (plus) |
|---|---|---|---|
| Papel | só câmera + Wi-Fi | câmera + Wi-Fi + Linux | câmera + Wi-Fi + **inferência embarcada** |
| Processamento | microcontrolador de 240 MHz, PSRAM de 4 MB | CPU ARM de 4 núcleos | CPU + NPU de **13 TOPS** |
| Roda o YOLOv5 do projeto? | não (memória insuficiente) | sim, devagar (ONNX na CPU) | sim, em tempo real (INT8) |
| Depende do PC no campo? | sim | sim (modo BASE) | **não**: só as detecções e os alertas trafegam |
| Quando faz sentido | protótipo barato, câmera fixa | câmera com mais controle e a mesma ótica do dataset | rover autônomo operando em campo |

**Decisão:** uso o modo BASE para demonstrar o fluxo pedido no enunciado com qualquer uma das câmeras.
O plus com Hailo trata a etapa seguinte do cenário da FarmTech: o rover reconhecer os objetos no pasto
sem um computador por perto. Escolhi o Hailo-8L por motivos técnicos:

- **Inferência local:** o modelo roda no próprio rover, sem depender do PC nem de conexão com a nuvem,
  e as imagens não precisam sair do dispositivo.
- **Latência:** a NPU executa o YOLOv5 quantizado em INT8 em poucos milissegundos por quadro, o que
  verifico na seção 8 comparando com a CPU do Pi.
- **Desempenho por consumo:** são 13 TOPS com consumo de poucos watts, compatível com a alimentação de
  um equipamento embarcado.
- **Rede:** como a inferência acontece no rover, o Wi-Fi deixa de ser crítico para a detecção; uma
  queda de sinal atrasa o envio dos alertas, mas não interrompe o reconhecimento.

Detalho essa escolha no vídeo complementar sobre a escolha do Hailo, ⏳ `https://youtu.be/<inserir>`.

### Limitações assumidas
- A latência fim a fim usa o relógio da câmera (`X-Timestamp`) e só é confiável quando a câmera e o PC
  estão sincronizados por NTP. Sem essa sincronização, a métrica de referência passa a ser o **RTT**
  medido pelo próprio cliente.
- O modelo foi treinado com cerca de 32 imagens por classe. Em vídeo ao vivo, espero detecções que
  oscilam entre quadros, principalmente de `escorpiao`.
- A quantização INT8 do Hailo pode reduzir a precisão nas classes pequenas. Em vez de supor esse efeito,
  eu o meço na seção 8.

## 3. Arquitetura

![Arquitetura do Ir Além](figuras/arquitetura.png)

*Figura de autoria própria (`figuras/arquitetura.svg`). A faixa verde mostra o fluxo pedido no enunciado
e a faixa azul, o plus embarcado. As duas trilhas chegam ao mesmo cliente no VS Code.*

## 4. Componentes

```
ir_alem/
├── camera_server.py          # câmera → Wi-Fi (Pi ou PC); modo PLUS com --inferencia hailo|onnx
├── cliente_vscode.py         # PC: recebe, detecta (best.pt), alerta e grava os logs
├── backends/                 # inferência embarcada com interface comum detect(frame) -> lista
│   ├── __init__.py           #   classes, cores e função anotar()
│   ├── yolo_utils.py         #   letterbox, decodificação do YOLOv5 e NMS em numpy
│   ├── onnx_backend.py       #   plano B: ONNX Runtime na CPU do Pi
│   └── hailo_backend.py      #   plus: HailoRT na NPU Hailo-8L
├── esp32cam/esp32cam_stream.ino   # ESP32-CAM servindo as mesmas rotas
├── hailo/README_conversao.md      # best.pt → ONNX → .hef
├── figuras/arquitetura.svg / .png
└── logs/                     # frames.csv e alertas.csv gerados pelo cliente
```

### 4.1 Transmissão: o mesmo contrato em todas as câmeras
Defini as mesmas rotas para todas as câmeras, de modo que o cliente não precisa saber qual hardware
está do outro lado:

| Rota | Conteúdo |
|---|---|
| `/stream` | MJPEG contínuo; cada quadro leva `X-Timestamp` (instante da captura) |
| `/frame.jpg` | um quadro |
| `/saude` | JSON de status, usado para medir o RTT |
| `/deteccoes` | JSON com as detecções (só no modo PLUS) |

No trecho abaixo, do `camera_server.py`, cada quadro novo libera todas as conexões abertas, e nenhuma
delas recebe o mesmo quadro duas vezes:
```python
with estado.cond:
    estado.cond.wait_for(lambda: estado.numero != visto, timeout=5)
    jpeg, visto, ts = estado.jpeg, estado.numero, estado.ts
self.wfile.write(f"--{FRONTEIRA}\r\nContent-Type: image/jpeg\r\n"
                 f"Content-Length: {len(jpeg)}\r\nX-Timestamp: {ts:.6f}\r\n\r\n".encode())
```

### 4.2 Recepção e reconhecimento: `cliente_vscode.py`
O cliente interpreta o multipart por conta própria, em vez de usar `cv2.VideoCapture`, para ter acesso
ao `X-Timestamp`. Se o Wi-Fi cair, ele tenta reconectar a cada 2 s sem travar a janela. No modo BASE, o
`best.pt` é carregado pelo hub do YOLOv5; o `trust_repo=True` evita a confirmação interativa que as
versões recentes do PyTorch pedem ao baixar o repositório:
```python
self.modelo = torch.hub.load("ultralytics/yolov5", "custom", path=str(caminho),
                                         trust_repo=True)
resultado = self.modelo(frame[:, :, ::-1])            # o hub espera RGB
```
Detecções de `cobra` e `escorpiao` acendem uma faixa vermelha de alerta e geram uma linha em
`alertas.csv`. Cada quadro gera uma linha em `frames.csv`, que é a base da análise das seções 6 e 7.

### 4.3 Inferência embarcada: `backends/`
Reimplementei em numpy o pós-processamento do YOLOv5 (letterbox, decodificação e NMS por classe) para
que o Pi execute a inferência **sem PyTorch**. O backend Hailo usa o NMS do próprio chip quando o `.hef`
é compilado com o config `yolov5s` do Model Zoo (detalhes em `hailo/README_conversao.md`). A opção
`--classes coco` permite validar o pipeline com um `.hef` pré-treinado no COCO enquanto o `best.hef`
do projeto não está pronto.

## 5. Como executar

**No PC (VS Code):** execute `pip install -r requirements-pc.txt` e copie o `best.pt` da Entrega 1 para `ir_alem/`.

| Cenário | Câmera | PC (terminal do VS Code) |
|---|---|---|
| BASE · webcam do PC | não se aplica | `python cliente_vscode.py --fonte 0 --modelo best.pt` |
| BASE · ESP32-CAM | gravar o `esp32cam_stream.ino` e ler o IP no Monitor Serial | `python cliente_vscode.py --fonte http://<IP>/stream --modelo best.pt` |
| BASE · Raspberry Pi 5 | `python camera_server.py --fonte picamera2` | `python cliente_vscode.py --fonte http://<IP>:8000/stream --modelo best.pt` |
| PLUS · Pi 5 + Hailo-8L | `python camera_server.py --fonte picamera2 --inferencia hailo --modelo best.hef` | `python cliente_vscode.py --fonte http://<IP>:8000/stream` |
| PLUS · plano B (CPU do Pi) | `python camera_server.py --fonte picamera2 --inferencia onnx --modelo best.onnx` | `python cliente_vscode.py --fonte http://<IP>:8000/stream` |
| Teste do pipeline sem `best.hef` | `python camera_server.py --fonte teste.mp4 --loop --inferencia hailo --modelo /usr/share/hailo-models/yolov8s_h8l.hef --classes coco` | `python cliente_vscode.py --fonte http://<IP>:8000/stream` |

Na janela, a tecla **s** salva o quadro atual em `prints/` e a tecla **q** encerra. Sem tela (SSH ou
testes), o cliente roda com `--sem-janela --max-frames 300`.

## 6. Protocolo de medição

Para comparar os cenários em condições equivalentes, fixei: **a mesma cena** (o pasto, ou a TV exibindo
imagens de cobra e escorpião, como o enunciado permite), **a mesma rede Wi-Fi**, **2 minutos por cenário**
e o cliente gravando em `logs/frames.csv`.

| Métrica | Como é medida | Coluna do log |
|---|---|---|
| FPS fim a fim | quadros exibidos por segundo no PC (janela de 1 s) | `fps` |
| Latência fim a fim | instante de exibição − `X-Timestamp` da captura (exige NTP) | `latencia_fim_a_fim_ms` |
| RTT de rede | ida e volta de `GET /saude`, a cada 100 ms | `rtt_rede_ms` |
| Latência de inferência | tempo do `detect()`: no PC (BASE) ou no Pi (PLUS) | `latencia_inferencia_ms` |
| Alertas | quadros com `cobra` ou `escorpiao` | `alerta` e `alertas.csv` |

**Hipóteses que pretendo verificar:**
1. No BASE, o gargalo é a inferência no PC sem GPU, e não o Wi-Fi.
2. No PLUS com Hailo, a inferência cai para poucos milissegundos e o FPS passa a ser limitado pela câmera e pela rede.
3. O plano B (ONNX na CPU do Pi) funciona, mas com FPS baixo demais para um rover em movimento.

In [ ]:
# Análise dos logs gravados pelo cliente_vscode.py (executado antes nos cenários da seção 6)
from pathlib import Path                                   # caminhos independentes do sistema operacional
import pandas as pd                                        # leitura e agregação dos logs
import matplotlib.pyplot as plt                            # gráficos da análise

LOGS = Path("logs")                                        # pasta padrão do cliente (opção --logs)
frames_csv, alertas_csv = LOGS / "frames.csv", LOGS / "alertas.csv"

if frames_csv.exists():
    frames = pd.read_csv(frames_csv)                       # uma linha por quadro exibido
    alertas = pd.read_csv(alertas_csv) if alertas_csv.exists() else pd.DataFrame()
    print(f"{len(frames)} quadros em {frames['sessao'].nunique()} sessão(ões); {len(alertas)} alertas")
    display(frames.tail())
else:
    frames, alertas = None, None
    print("⏳ Ainda não há logs/frames.csv: execute o cliente_vscode.py nos cenários da seção 6.")

In [ ]:
# Resumo por modo: mediana e percentil 95 de cada métrica
if frames is not None:
    def p95(serie):                                        # percentil 95: o pior caso que ainda ocorre com frequência
        return serie.quantile(0.95)

    metricas = ["fps", "latencia_fim_a_fim_ms", "rtt_rede_ms", "latencia_inferencia_ms"]
    resumo = (frames[frames["frame"] > 30]                 # descarta o aquecimento (primeiros 30 quadros, cerca de 1 s)
              .groupby("modo")[metricas]
              .agg(["median", p95])
              .round(1))
    display(resumo)

In [ ]:
# Gráficos: distribuição das latências e FPS ao longo do tempo, por modo
if frames is not None:
    dados = frames[frames["frame"] > 30].copy()
    dados["t"] = dados.groupby("sessao")["timestamp"].transform(lambda s: s - s.min())   # segundos desde o início da sessão

    fig, eixos = plt.subplots(1, 3, figsize=(16, 4.5))
    modos = sorted(dados["modo"].unique())
    for eixo, coluna, titulo in [(eixos[0], "latencia_inferencia_ms", "Inferência (ms)"),
                                 (eixos[1], "latencia_fim_a_fim_ms", "Fim a fim (ms)")]:
        series = [dados.loc[dados["modo"] == m, coluna].dropna() for m in modos]
        eixo.boxplot(series, showfliers=False)             # caixa = quartis; outliers ocultos
        eixo.set_xticks(range(1, len(modos) + 1), modos)   # nomes dos modos no eixo x
        eixo.set_title(titulo); eixo.grid(axis="y", alpha=0.3)
    for m in modos:                                        # FPS ao longo do tempo, uma linha por modo
        sub = dados[dados["modo"] == m]
        eixos[2].plot(sub["t"], sub["fps"], label=m, lw=1)
    eixos[2].set_title("FPS no tempo"); eixos[2].set_xlabel("s"); eixos[2].legend(); eixos[2].grid(alpha=0.3)
    plt.tight_layout(); plt.show()

In [ ]:
# Alertas de risco biológico: quantidade, classe e confiança
if alertas is not None and len(alertas):
    display(alertas.groupby("classe")["conf"].agg(["count", "mean", "min", "max"]).round(3))
    alertas["conf"].plot.hist(bins=20, title="Confiança dos alertas", figsize=(6, 3.5))
    plt.xlabel("confiança"); plt.show()
else:
    print("⏳ Ainda não há alertas registrados.")

## 7. Evidências

> ⏳ Pendente de execução: após rodar cada cenário da tabela da seção 5, insiro aqui um print salvo com
> a tecla **s** (`prints/`) e comento o que ele mostra (caixas detectadas, FPS e latências exibidas na janela).

| Cenário | Print | Observação |
|---|---|---|
| BASE · webcam do PC | ⏳ `prints/...jpg` | |
| BASE · ESP32-CAM | ⏳ `prints/...jpg` | |
| BASE · Raspberry Pi 5 | ⏳ `prints/...jpg` | |
| PLUS · Hailo-8L | ⏳ `prints/...jpg` | |
| Alerta de cobra (TV) | ⏳ `prints/...jpg` | |

No vídeo de demonstração, também abro a rota `/saude` no navegador do celular, o que comprova que o
dispositivo está acessível pela rede Wi-Fi.

## 8. Plus: Raspberry Pi 5 + Hailo-8L

A conversão `best.pt → ONNX → .hef` está descrita em `hailo/README_conversao.md`.

Comparo o **benchmark só da NPU** (`hailortcli benchmark best.hef`) com as medições **fim a fim** (logs da seção 6):

| | FPS | Latência de inferência (ms) | Observação |
|---|---|---|---|
| `hailortcli benchmark` (só NPU) | ⏳ | ⏳ | limite teórico do chip |
| PLUS Hailo, fim a fim | ⏳ | ⏳ | + captura + JPEG + Wi-Fi |
| PLUS ONNX na CPU do Pi | ⏳ | ⏳ | plano B |
| BASE no PC (`best.pt`) | ⏳ | ⏳ | referência |

> ⏳ Pendente de execução: com o benchmark e as sessões do cliente, comparo o FPS do chip isolado com o
> FPS fim a fim. A diferença indica quanto do tempo vai para captura, codificação JPEG e Wi-Fi.

**Efeito da quantização INT8** (mesmo conjunto de teste da Entrega 1):

| Classe | mAP@0.5 FP32 (Colab) | mAP@0.5 INT8 (Hailo) | Δ |
|---|---|---|---|
| vaca | ⏳ | ⏳ | |
| cobra | ⏳ | ⏳ | |
| escorpiao | ⏳ | ⏳ | |
| arvore | ⏳ | ⏳ | |
| cerca | ⏳ | ⏳ | |

> Hipótese: o INT8 penaliza mais as classes de objeto pequeno (`escorpiao`, `cobra`). Se isso se
> confirmar, o rover vai precisar de mais resolução de entrada ou de um segundo estágio de classificação
> para as classes de segurança.

## 9. Conclusões

> ⏳ Pendente de execução: completo esta seção com os números medidos nas seções 6 a 8.

- **Funcionalidade:** a câmera coleta as imagens, o Wi-Fi as transmite e o VS Code reconhece os objetos
  com o `best.pt` da Entrega 1, com os três tipos de câmera e o mesmo cliente.
- **Wi-Fi:** ⏳ RTT mediano de __ ms e latência fim a fim p95 de __ ms; avalio também se o cliente se
  recuperou sozinho das quedas de sinal.
- **Plus:** ⏳ com o Hailo-8L, a inferência passou de __ ms (CPU do Pi) para __ ms, e o rover deixa de
  depender do PC no campo.
- **Limitações:** dataset pequeno (cerca de 32 imagens de treino por classe), latência fim a fim
  dependente de NTP, possível perda de precisão com INT8 nas classes pequenas e testes de cobra e
  escorpião feitos com imagens na TV, não com animais reais.